# ASD-XMF R1 Photo Data Gaze
CPU: rebuild the XMF-Photo dataset (D1), near-duplicate audit of its test sets, gaze study with 10 CV repeats and the gaze-vs-recording-quality checks.

Each section is one former notebook, run as its own process. A section waits for the ones it needs; a time guard stops before Kaggle's 12-hour limit and the next run continues where this one stopped.

| section | source | needs |
|---|---|---|
| `d1` | `d1_photo_builder.py` | - |
| `dup` | `dup_photo_near_duplicates.py` | d1 |
| `nb13` | `nb13_gaze_eyetracking.py` | - |
| `g1` | `g1_gaze_robustness.py` | nb13 |

In [ ]:
RUN_PROFILE = "full"   # "full" on GPU; "preflight" = quick CPU smoke test (tools/push.py)
PREFLIGHT = RUN_PROFILE == "preflight"
import os, sys, glob, json, time, warnings
warnings.filterwarnings("ignore")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

def _find_lib():
    for c in ("/kaggle/input/asd-xmf-lib", "/kaggle/input/datasets/tahidulkabir/asd-xmf-lib"):
        if os.path.exists(os.path.join(c, "asd_utils.py")):
            return c
    for depth in range(1, 6):
        hits = glob.glob("/kaggle/input/" + "*/" * depth + "asd_utils.py")
        if hits:
            return os.path.dirname(hits[0])
    return os.path.abspath(os.path.join(os.getcwd(), "..", "..", "asd_lib"))  # local run

def _require_internet():
    import socket
    try:
        socket.create_connection(("pypi.org", 443), timeout=10).close()
    except OSError:
        raise SystemExit("NO INTERNET in this Kaggle session: verify the account's phone number (Kaggle > Settings) "
                         "and keep 'Internet on'. Stopping so no step runs with missing models or features.")
_require_internet()

LIB = _find_lib()
sys.path.insert(0, LIB)
import asd_utils as U
print("asd_lib:", LIB, "| profile:", RUN_PROFILE)
ENV = U.env_report()
open(os.path.join(str(U.WORK), "_run_profile_" + RUN_PROFILE + ".txt"), "w").write(time.strftime("%Y-%m-%d %H:%M:%S"))  # queue marker

In [ ]:
import asd_runner as R
R.setup('tahidulkabir/asd-xmf-r1-photo-data-gaze', RUN_PROFILE, [{"name": "d1", "after": [], "rev": 1}, {"name": "dup", "after": ["d1"], "rev": 1}, {"name": "nb13", "after": [], "rev": 1}, {"name": "g1", "after": ["nb13"], "rev": 1}], hours=11.0, rev=1)

## Section `d1`: D1 - XMF-Photo: unified facial-expression image dataset

In [ ]:
%%writefile {R.SEC_DIR}/d1.py
# # D1 - XMF-Photo: unified facial-expression image dataset
# Combines FER2013, RAF-DB and CK+ into one dataset with one 7-class label space
# (neutral, happy, sad, surprise, fear, disgust, anger) for pre-training the face branch (plan 4.2 / 5.1).
#
# * decoded once to 160 px RGB uint8 (`images.npy`), one row per image in `manifest.csv`
# * official FER2013 / RAF-DB test splits are kept as test; 10 % of train becomes validation
# * **CK+ is an external test set only** (its frames come in per-person sequences without subject IDs,
#   so a random split would leak)
# * perceptual-hash de-duplication across all sources: a training image whose hash equals a
#   test image is removed from training
# * the scraped "Autistic Children Facial Data Set" and its mirrors are excluded (plan section 1)
# 

import numpy as np, pandas as pd, cv2
from pathlib import Path
import multiprocessing as mp
from functools import partial
import asd_models as M
IMG = 160
OUT = U.WORK / "xmf_photo"; OUT.mkdir(parents=True, exist_ok=True)
CLS = M.EXPR_CLASSES
FER = {"neutral": "neutral", "happy": "happy", "sad": "sad", "surprise": "surprise", "fear": "fear", "disgust": "disgust", "angry": "anger"}
RAF = {"1": "surprise", "2": "fear", "3": "disgust", "4": "happy", "5": "sad", "6": "anger", "7": "neutral"}
CKP = {0: "anger", 1: "disgust", 2: "fear", 3: "happy", 4: "sad", 5: "surprise", 6: "neutral"}  # 7 = contempt -> dropped

rows = []
def add_folder(root, mapping, source):
    for split in ("train", "test"):
        for cdir in (Path(root) / split).iterdir() if (Path(root) / split).exists() else []:
            lab = mapping.get(cdir.name.lower())
            if lab and cdir.is_dir():
                rows.extend({"path": str(f), "label": CLS.index(lab), "source": source, "orig_split": split}
                            for f in cdir.iterdir() if f.suffix.lower() in (".jpg", ".jpeg", ".png"))
add_folder(U.find_path("angry", want="dir").parent.parent, FER, "fer2013")
add_folder(U.find_path("DATASET", want="dir"), RAF, "rafdb")
ck = pd.read_csv(U.find_path("ckextended.csv"))
ck = ck[ck.emotion.isin(CKP)]
df = pd.DataFrame(rows)
print(df.groupby(["source", "orig_split"]).size(), "\nCK+ images:", len(ck))


# ## Decode once (all CPU cores)
# 

if PREFLIGHT:
    df = pd.concat([g.head(300) for _, g in df.groupby(["source", "orig_split"])]).reset_index(drop=True)
    ck = ck.head(100)
with mp.get_context("spawn").Pool(os.cpu_count()) as pool:
    imgs = list(pool.imap(partial(U.load_image_rgb, size=IMG), df.path.tolist(), chunksize=256))
ck_imgs = [cv2.cvtColor(cv2.resize(np.array(p.split(), np.uint8).reshape(48, 48), (IMG, IMG), interpolation=cv2.INTER_CUBIC), cv2.COLOR_GRAY2RGB)
           for p in ck.pixels]
ckdf = pd.DataFrame({"path": [f"ckplus/{i}" for i in ck.index], "label": [CLS.index(CKP[e]) for e in ck.emotion],
                     "source": "ckplus", "orig_split": "test"})
df = pd.concat([df, ckdf], ignore_index=True)
X = np.stack(imgs + ck_imgs)
del imgs, ck_imgs  # keep peak memory low (~2 copies of a 4 GB array at most)
print("decoded", X.shape)


# ## Splits + cross-source pHash de-duplication
# 

rng = np.random.default_rng(0)
df["split"] = np.where(df.orig_split == "test", "test", "train")
tr = df.index[df.split == "train"].to_numpy()
df.loc[rng.choice(tr, int(0.10 * len(tr)), replace=False), "split"] = "val"
df["phash"] = [U.phash(im) for im in X]
# Vectorised de-duplication on identical pHash (seconds for ~52k images; a pairwise Hamming search
# in Python does not finish in a 12 h session).  Priority test > val > train: a hash that occurs in
# test is removed from val/train, one that occurs in val is removed from train, then one copy per split.
pri = df.split.map({"test": 0, "val": 1, "train": 2})
first = df.assign(pri=pri).sort_values("pri").drop_duplicates("phash")
best_split = first.set_index("phash").split
keep_mask = (df.split.values == df.phash.map(best_split).values) & ~df.duplicated(["phash", "split"]).values
print(f"duplicate (identical pHash) images removed: {int((~keep_mask).sum())}")
df, X = df[keep_mask].reset_index(drop=True), X[keep_mask]


np.save(OUT / "images.npy", X)
np.save(OUT / "labels.npy", df.label.values.astype(np.int64))
np.save(OUT / "split.npy", df.split.values.astype(object))
np.save(OUT / "source.npy", df.source.values.astype(object))
df.drop(columns=["path"]).assign(label_name=df.label.map(dict(enumerate(CLS)))).to_csv(OUT / "manifest.csv", index=False)
summary = df.groupby(["source", "split"]).size().unstack(fill_value=0)
print(summary)
(OUT / "DATASHEET.md").write_text(f"""# XMF-Photo
Unified 7-class facial-expression dataset ({', '.join(CLS)}), {len(df)} images at {IMG} px RGB.
Sources: FER2013 (msambare/fer2013), RAF-DB (shuvoalok/raf-db-dataset), CK+ (davilsena/ckdataset; external test only).
Splits: official test sets kept; 10 % of training as validation; images with identical pHash removed (test > val > train priority).
Use: pre-training of the face-expression encoder. Not an autism dataset. Licences: as stated by each original source (research use).

{summary.to_markdown()}
""")


In [ ]:
R.run('d1')

## Section `dup`: DUP - Near-duplicate audit of the XMF-Photo test sets

In [ ]:
%%writefile {R.SEC_DIR}/dup.py
# # DUP - Near-duplicate audit of the XMF-Photo test sets
# D1 removes test images whose pHash is *identical* to a training image.  Re-encoded, resized or slightly
# cropped copies differ in a few pHash bits and survive that rule.  FER2013 and RAF-DB carry no person IDs,
# so a person-level split is impossible; the closest check is this exhaustive near-duplicate search.
#
# For every test image: the smallest Hamming distance (0-64 bits) between its 64-bit pHash and any train or
# validation image (exact, all pairs).  R2 re-scores every model on the "clean" test set without images
# within `CLEAN_MAX_DIST` bits of a training image, so the reader sees how much near-copies inflate accuracy.
# 

import numpy as np, pandas as pd
OUT = U.WORK / "dup"; OUT.mkdir(parents=True, exist_ok=True)
SRC = U.WORK / "xmf_photo" if (U.WORK / "xmf_photo" / "manifest.csv").exists() else U.find_path("images.npy").parent
# <= 4 of 64 bits: in the first full run ~85 % of test images at distance 2-4 shared the label of their nearest
# training image (true copies), at 6 bits 70 %, at 8 bits 46 % (near the ~25 % of unrelated faces) -
# small 48-px faces look alike to pHash, so wider thresholds catch look-alikes, not copies.  6 = sensitivity check.
CLEAN_MAX_DIST = 4
man = pd.read_csv(SRC / "manifest.csv", dtype={"phash": str})
split = np.load(SRC / "split.npy", allow_pickle=True)
assert len(man) == len(split), "manifest and split.npy differ"
h = np.array([int(x) for x in man.phash], dtype=np.uint64)
te = np.flatnonzero(split == "test")          # the same order photo_task uses for test predictions
ref = np.flatnonzero(split != "test")         # train + validation
print(f"test images: {len(te)}  | train+val images: {len(ref)}")

try:
    popcount = np.bitwise_count                # numpy >= 2.0
except AttributeError:
    _LUT = np.array([bin(i).count("1") for i in range(256)], dtype=np.uint8)
    popcount = lambda a: _LUT[a.view(np.uint8)].reshape(*a.shape, 8).sum(-1)

min_d = np.empty(len(te), dtype=np.int16); nn_row = np.empty(len(te), dtype=np.int64)
hr = h[ref]
for s in range(0, len(te), 256):
    d = popcount(h[te[s:s + 256], None] ^ hr[None, :]).astype(np.int16)
    j = d.argmin(1)
    min_d[s:s + 256] = d[np.arange(len(j)), j]; nn_row[s:s + 256] = ref[j]
res = pd.DataFrame({"test_pos": np.arange(len(te)), "row": te, "source": man.source.values[te],
                    "label": man.label.values[te], "min_dist_trainval": min_d, "nearest_row": nn_row,
                    "nearest_same_label": man.label.values[nn_row] == man.label.values[te]})
res.to_csv(OUT / "near_dup_test.csv", index=False)


rows = []
for src, g in [("all", res)] + list(res.groupby("source")):
    r = {"source": src, "n_test": len(g)}
    for t in (0, 2, 4, 6, 8, 10, 12):
        r[f"le_{t}"] = int((g.min_dist_trainval <= t).sum())
    r[f"pct_le_{CLEAN_MAX_DIST}"] = round(100 * (g.min_dist_trainval <= CLEAN_MAX_DIST).mean(), 2)
    rows.append(r)
summ = pd.DataFrame(rows)
print(U.table_md(summ))
summ.to_csv(OUT / "near_dup_summary.csv", index=False)
# how often a test image and its nearest training image share the expression label, by distance
# (true copies mostly share it; unrelated faces only by chance) - this is what sets CLEAN_MAX_DIST
by_d = res.groupby("min_dist_trainval").agg(n=("row", "size"), same_label=("nearest_same_label", "mean")).reset_index()
chance = float((man.label.value_counts(normalize=True) ** 2).sum())
by_d["chance_same_label"] = round(chance, 3)
print(U.table_md(by_d[by_d.min_dist_trainval <= 14]))
by_d.to_csv(OUT / "near_dup_label_agreement.csv", index=False)
U.save_json({"clean_max_dist": CLEAN_MAX_DIST, "n_test": len(te),
             "n_near_dup": int((min_d <= CLEAN_MAX_DIST).sum())}, OUT / "near_dup_config.json")


In [ ]:
R.run('dup')

## Section `nb13`: NB13 - Gaze module + external sanity check (Cilia et al. 2022 eye-tracking data)

In [ ]:
%%writefile {R.SEC_DIR}/nb13.py
# # NB13 - Gaze module + external sanity check (Cilia et al. 2022 eye-tracking data)
# Plan section 4.2: *"Gaze-feature module, external sanity check."*  59 children
# (29 ASD, 30 TD, CARS-scored) watched video stimuli on an SMI eye tracker.
# This is the one public ASD-vs-TD set with consent and clinical labels that we can
# analyse **today**, so it also rehearses the full evaluation protocol of section 7:
#
# * child-level features only, repeated stratified 5-fold CV by child (3 seeds)
# * 95 % bootstrap CIs, DeLong vs baselines, calibration, decision curves
# * **shortcut baselines**: sex + age only, and data-quality only (tracking ratio).
#   In this cohort ASD is 83 % male vs 43 % in TD, so sex alone predicts the label -
#   the gaze model must beat that, and we also report a males-only analysis.
# * SHAP on gaze features (behaviour-level XAI)
#
# Cite: Cilia F. et al., *Eye-tracking dataset to support the research on autism spectrum
# disorder*, 2022 (figshare / Kaggle mirror `imtkaggleteam/eye-tracking-autism`).
# 

import numpy as np, pandas as pd
from pathlib import Path
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
OUT = U.WORK / "nb13"; OUT.mkdir(parents=True, exist_ok=True)
SEEDS = tuple(range(int(globals().get("VARIANT", {}).get("n_seeds", 3))))  # R1 repeats the child-wise CV 10 times
N_BOOT = 300 if PREFLIGHT else 2000

meta_p = U.find_path("Metadata_Participants.csv")
csv_dir = U.find_path("Eye-tracking Output", want="dir")
meta = pd.read_csv(meta_p)
meta.columns = [c.strip() for c in meta.columns]
meta = meta.rename(columns={"ParticipantID": "pid", "Gender": "sex", "Age": "age", "Class": "cls", "CARS Score": "cars"})
meta["label"] = (meta.cls.str.upper() == "ASD").astype(int)
print(meta.groupby("cls").agg(n=("pid", "size"), male=("sex", lambda s: (s == "M").mean()), age=("age", "mean")))


# ## 1. Event-level parsing -> child-level gaze features
# 

USE = ["RecordingTime [ms]", "Stimulus", "Trial", "Participant", "Tracking Ratio [%]", "Category Right", "Index Right",
       "Pupil Diameter Right [mm]", "Pupil Diameter Left [mm]", "Point of Regard Right X [px]",
       "Point of Regard Right Y [px]", "AOI Name Right"]
files = sorted(csv_dir.glob("*.csv"), key=lambda p: int(p.stem) if p.stem.isdigit() else 0)
if PREFLIGHT:
    files = files[:4]
parts = []
for f in files:
    d = pd.read_csv(f, usecols=lambda c: c in USE, low_memory=False)
    d["file"] = f.stem
    parts.append(d)
raw = pd.concat(parts, ignore_index=True)
raw = raw[pd.to_numeric(raw.Participant, errors="coerce").notna()].copy()
raw["pid"] = raw.Participant.astype(float).astype(int)
for c in ["RecordingTime [ms]", "Tracking Ratio [%]", "Pupil Diameter Right [mm]", "Pupil Diameter Left [mm]",
          "Point of Regard Right X [px]", "Point of Regard Right Y [px]", "Index Right"]:
    raw[c] = pd.to_numeric(raw[c], errors="coerce")
raw["stim"] = raw.Stimulus.astype(str).str.lower().str.replace(r"\.(avi|mp4|jpg|png)$", "", regex=True).str.strip()
print(raw.shape, raw.pid.nunique(), "participants;", raw.stim.nunique(), "stimuli")

def child_features(g: pd.DataFrame) -> dict:
    t = g["RecordingTime [ms]"]
    dur_min = max(1e-3, (t.max() - t.min()) / 60000)
    cat = g["Category Right"].astype(str)
    ev = g.assign(cat=cat).groupby(["Trial", "Index Right", "cat"])["RecordingTime [ms]"].agg(["min", "max", "size"]).reset_index()
    fix = ev[ev.cat == "Fixation"]
    sac = ev[ev.cat == "Saccade"]
    fx_d = (fix["max"] - fix["min"]).clip(lower=0)
    por = g.loc[cat == "Fixation", ["Point of Regard Right X [px]", "Point of Regard Right Y [px]"]].dropna()
    pup = g[["Pupil Diameter Right [mm]", "Pupil Diameter Left [mm]"]].replace(0, np.nan).mean(1)
    aoi = g["AOI Name Right"].astype(str)
    on_aoi = (aoi != "-") & (aoi != "nan") & (aoi.str.lower() != "white space")
    return {
        "fix_prop": (cat == "Fixation").mean(), "sac_prop": (cat == "Saccade").mean(), "blink_prop": (cat == "Blink").mean(),
        "fix_rate": len(fix) / dur_min, "sac_rate": len(sac) / dur_min,
        "blink_rate": ev[ev.cat == "Blink"].shape[0] / dur_min,
        "fix_dur_mean": fx_d.mean(), "fix_dur_median": fx_d.median(), "fix_dur_sd": fx_d.std(),
        "gaze_disp_x": por.iloc[:, 0].std() if len(por) else np.nan, "gaze_disp_y": por.iloc[:, 1].std() if len(por) else np.nan,
        "pupil_mean": pup.mean(), "pupil_sd": pup.std(), "aoi_on_prop": on_aoi.mean(),
        "track_ratio": g["Tracking Ratio [%]"].mean(),
    }

feats_all = raw.groupby("pid").apply(child_features).apply(pd.Series)
feats_all.columns = ["all_" + c for c in feats_all.columns]
# per-stimulus features for stimuli seen by >= 80 % of children
cover = raw.groupby("stim").pid.nunique()
common = cover[cover >= 0.8 * raw.pid.nunique()].index.tolist()[:12]
print("common stimuli:", common)
per = []
for s in common:
    f = raw[raw.stim == s].groupby("pid").apply(child_features).apply(pd.Series)
    f = f[["fix_prop", "fix_dur_mean", "gaze_disp_x", "gaze_disp_y", "aoi_on_prop", "blink_rate"]]
    f.columns = [f"{s[:24]}__{c}" for c in f.columns]
    per.append(f)
F = pd.concat([feats_all] + per, axis=1)
data = meta.set_index("pid").join(F, how="inner").reset_index()
QUALITY = ["all_track_ratio"]
GAZE = [c for c in F.columns if c not in QUALITY]
print("children with gaze:", len(data), "| gaze features:", len(GAZE))
data.to_csv(OUT / "gaze_child_features.csv", index=False)


# ## 2. Repeated child-wise CV for gaze models and shortcut baselines
# 

from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.svm import SVC
data["child_id"] = data.pid.astype(str)
data["sex_m"] = (data.sex == "M").astype(int)
splits = U.make_child_splits(data[["child_id", "label", "sex"]], n_splits=5, seeds=SEEDS, test_frac=0.0)
data = data.merge(splits[["child_id", "split"] + [f"fold_s{s}" for s in SEEDS]], on="child_id")

def lr(C=0.3): return make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), LogisticRegression(C=C, max_iter=5000))
MODELS = {
    "gaze_logreg": (GAZE, lambda: lr(0.1)),
    "gaze_svm": (GAZE, lambda: make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), SVC(C=1.0, probability=True))),
    "gaze_rf": (GAZE, lambda: make_pipeline(SimpleImputer(strategy="median"), RandomForestClassifier(500, min_samples_leaf=2, random_state=0))),
    "gaze_et": (GAZE, lambda: make_pipeline(SimpleImputer(strategy="median"), ExtraTreesClassifier(500, min_samples_leaf=2, random_state=0))),
    "gaze_plus_demo_logreg": (GAZE + ["sex_m", "age"], lambda: lr(0.1)),
    "SHORTCUT_sex_age": (["sex_m", "age"], lambda: lr(1.0)),
    "SHORTCUT_quality": (QUALITY, lambda: lr(1.0)),
}
try:
    import lightgbm as lgb
    MODELS["gaze_lgbm"] = (GAZE, lambda: lgb.LGBMClassifier(n_estimators=300, learning_rate=0.03, num_leaves=7,
                                                            min_child_samples=5, subsample=0.8, colsample_bytree=0.7, verbose=-1))
except Exception:
    pass

oof_rows = []
for name, (cols, mk) in MODELS.items():
    for s in SEEDS:
        for f in sorted(data[f"fold_s{s}"].unique()):
            tr, va = data[data[f"fold_s{s}"] != f], data[data[f"fold_s{s}"] == f]
            U.assert_no_leak(tr.child_id, va.child_id)
            m = mk().fit(tr[cols], tr.label)
            p = m.predict_proba(va[cols])[:, 1]
            d = pd.DataFrame({"child_id": va.child_id, "y_true": va.label, "prob": p, "part": "val"})
            U.save_preds(d, name, f, OUT / "preds", seed=s)
            oof_rows.append(d.assign(model=name, seed=s, fold=f))
oof = pd.concat(oof_rows)


# ## 3. Results with 95 % CIs (pooled over 3 seeds, threshold at 85 % sensitivity)
# 

summ, curves = [], {}
for name in MODELS:
    s = U.summarise_model(oof[oof.model == name], target_sens=0.85, n_boot=N_BOOT)
    ci = s["oof_ci"].set_index("metric")
    row = {"model": name, "n_children": s["oof"]["n"]}
    for k in ("auroc", "auprc", "sens", "spec", "bacc", "brier", "ece"):
        row[k] = f"{ci.loc[k,'est']:.3f} [{ci.loc[k,'lo']:.3f}-{ci.loc[k,'hi']:.3f}]"
    summ.append(row)
    curves[name] = oof[oof.model == name].groupby("child_id").agg(y=("y_true", "first"), p=("prob", "mean"))
summ = pd.DataFrame(summ)
print(U.table_md(summ))
summ.to_csv(OUT / "gaze_results_ci.csv", index=False)

# DeLong: every model vs the sex+age shortcut, Holm-corrected
ref = curves["SHORTCUT_sex_age"]
tests = []
for name, c in curves.items():
    if name == "SHORTCUT_sex_age":
        continue
    j = c.join(ref, rsuffix="_ref")
    tests.append({"model": name, **U.delong_test(j.y, j.p, j.p_ref)})
tests = pd.DataFrame(tests)
tests["p_holm"] = U.holm(tests.p.values)
print(U.table_md(tests))
tests.to_csv(OUT / "gaze_delong_vs_sex_age.csv", index=False)


# ## 4. Confound control: males only, and fairness by sex / age
# 

best = max([m for m in curves if m.startswith("gaze_")], key=lambda m: U._safe_auc(curves[m].y, curves[m].p))
cb = curves[best].join(data.set_index("child_id")[["sex", "age"]])
male = cb[cb.sex == "M"]
print(f"best gaze model: {best}")
print("males only AUROC:", U.fmt_ci(U.bootstrap_ci(male.y.values, male.p.values, n_boot=N_BOOT)), f"(n={len(male)})")
cb["age_band"] = pd.cut(cb.age, [0, 6, 9, 20], labels=["<6y", "6-9y", "9y+"])
fair = U.fairness_table(cb.rename(columns={"y": "y_true", "p": "prob"}), ["sex", "age_band"], thr=0.5)
print(U.table_md(fair))
fair.to_csv(OUT / "gaze_fairness.csv", index=False)


# ## 5. Calibration, decision curve, ROC figure
# 

from sklearn.metrics import roc_curve
fig, ax = plt.subplots(1, 3, figsize=(15, 4.2))
for name in [best, "gaze_plus_demo_logreg", "SHORTCUT_sex_age", "SHORTCUT_quality"]:
    c = curves[name]
    fpr, tpr, _ = roc_curve(c.y, c.p)
    ax[0].plot(fpr, tpr, label=f"{name} ({U._safe_auc(c.y, c.p):.2f})")
    rc = U.reliability_curve(c.y, c.p, 5)
    ax[1].plot(rc.p_mean, rc.y_mean, "o-", label=name)
    dc = U.decision_curve(c.y, c.p)
    ax[2].plot(dc.thr, dc.nb_model, label=name)
ax[0].plot([0, 1], [0, 1], "k:"); ax[0].set(title="ROC (OOF, child-level)", xlabel="1 - specificity", ylabel="sensitivity")
ax[1].plot([0, 1], [0, 1], "k:"); ax[1].set(title="Reliability", xlabel="predicted", ylabel="observed")
ax[2].plot(dc.thr, dc.nb_all, "k--", label="refer all"); ax[2].axhline(0, color="k", lw=0.5)
ax[2].set(title="Decision curve", xlabel="threshold probability", ylabel="net benefit", ylim=(-0.1, 0.6))
for a in ax: a.legend(fontsize=7)
plt.tight_layout(); plt.savefig(OUT / "gaze_roc_calib_dca.png", dpi=200); plt.show()


# ## 6. SHAP on gaze features (behaviour-level explanation)
# 

try:
    import shap
    cols = GAZE
    model = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), LogisticRegression(C=0.1, max_iter=5000)).fit(data[cols], data.label)
    Xt = model[:-1].transform(data[cols])
    ex = shap.LinearExplainer(model[-1], Xt)
    sv = ex.shap_values(Xt)
    imp = pd.DataFrame({"feature": cols, "mean_abs_shap": np.abs(sv).mean(0), "coef": model[-1].coef_[0]}).sort_values("mean_abs_shap", ascending=False)
    print(U.table_md(imp.head(15)))
    imp.to_csv(OUT / "gaze_shap_importance.csv", index=False)
    shap.summary_plot(sv, pd.DataFrame(Xt, columns=cols), show=False, max_display=15)
    plt.tight_layout(); plt.savefig(OUT / "gaze_shap_summary.png", dpi=200); plt.show()
except Exception as e:
    print("SHAP skipped:", e)


U.save_json({"best_model": best, "n_children": int(len(data)), "n_gaze_features": len(GAZE),
             "note": "Repeated 5-fold CV by child over all 59 children (no separate test set: n too small)."},
            OUT / "nb13_summary.json")
print(sorted(p.name for p in OUT.iterdir()))


In [ ]:
R.run('nb13', {'n_seeds': 10})

## Section `g1`: G1 - Is the gaze signal more than recording quality?

In [ ]:
%%writefile {R.SEC_DIR}/g1.py
# # G1 - Is the gaze signal more than recording quality?
# In NB13 the tracking ratio alone (a data-quality measure) reached AUROC 0.84, close to the gaze models.
# A reviewer will ask whether the gaze models only re-learn quality.  Five checks on the same child-level
# features (one row per child, repeated stratified 5-fold CV over children, 10 repeats):
#
# 1. quality only vs gaze only vs gaze + quality (DeLong: does gaze add to quality?)
# 2. gaze features residualised on tracking ratio inside each training fold (quality removed)
# 3. high-quality children only (tracking ratio >= median), where quality varies little
# 4. label-permutation test of the gaze model (200 permutations)
# 5. how strongly each gaze feature correlates with tracking ratio
# 

import numpy as np, pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, LinearRegression
OUT = U.WORK / "g1"; OUT.mkdir(parents=True, exist_ok=True)
_own = U.WORK / "nb13" / "gaze_child_features.csv"   # written by section nb13 of the same notebook
data = pd.read_csv(_own if _own.exists() else U.find_path("gaze_child_features.csv"))
REPEATS = 3 if PREFLIGHT else 10
N_PERM = 20 if PREFLIGHT else 200
N_BOOT = 300 if PREFLIGHT else 2000
Q = "all_track_ratio"
skip = {"pid", "sex", "age", "cls", "cars", "label", "child_id", "split", "sex_m", Q}
GAZE = [c for c in data.columns if c not in skip and not c.startswith("fold_") and pd.api.types.is_numeric_dtype(data[c])]
y = data.label.values.astype(int)
print(f"{len(data)} children ({y.sum()} ASD) | {len(GAZE)} gaze features | quality = {Q}")

def lr(C):
    return make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), LogisticRegression(C=C, max_iter=5000))

def residualise(tr_df, te_df, cols):
    """Remove the linear effect of tracking ratio from every gaze feature, fitted on the training fold only."""
    q_tr = tr_df[[Q]].fillna(tr_df[Q].median()).values; q_te = te_df[[Q]].fillna(tr_df[Q].median()).values
    a, b = tr_df[cols].copy(), te_df[cols].copy()
    for c in cols:
        ok = a[c].notna().values
        if ok.sum() < 5:
            continue
        reg = LinearRegression().fit(q_tr[ok], a[c].values[ok])
        a[c] = a[c] - reg.predict(q_tr); b[c] = b[c] - reg.predict(q_te)
    return a, b

def cv_prob(df, cols, C=0.1, resid=False, repeats=REPEATS, labels=None):
    """Child-level probability averaged over `repeats` repetitions of stratified 5-fold CV."""
    lab = df.label.values.astype(int) if labels is None else labels
    P = np.zeros(len(df))
    for r in range(repeats):
        for tr, te in StratifiedKFold(5, shuffle=True, random_state=r).split(df, lab):
            a, b = (residualise(df.iloc[tr], df.iloc[te], cols) if resid else (df.iloc[tr][cols], df.iloc[te][cols]))
            P[te] += lr(C).fit(a, lab[tr]).predict_proba(b)[:, 1] / repeats
    return P


# ## 1-2. Quality, gaze, gaze + quality, gaze with quality removed
# 

probs = {
    "quality_only": cv_prob(data, [Q], C=1.0),
    "gaze_only": cv_prob(data, GAZE),
    "gaze_plus_quality": cv_prob(data, GAZE + [Q]),
    "gaze_quality_residualised": cv_prob(data, GAZE, resid=True),
}
rows = []
for name, p in probs.items():
    ci = U.bootstrap_ci(y, p, n_boot=N_BOOT)
    rows.append({"analysis": "all children", "model": name, "n": len(y), "auroc": U.fmt_ci(ci)})
tests = []
for a, b in [("gaze_plus_quality", "quality_only"), ("gaze_only", "quality_only"),
             ("gaze_quality_residualised", "quality_only"), ("gaze_quality_residualised", "gaze_only")]:
    tests.append({"model": a, "vs": b, **U.delong_test(y, probs[a], probs[b])})


# ## 3. High-quality children only
# 

hq = data[data[Q] >= data[Q].median()].reset_index(drop=True)
yh = hq.label.values.astype(int)
print(f"high-quality subset: {len(hq)} children ({yh.sum()} ASD), tracking ratio >= {data[Q].median():.1f}")
if min(yh.sum(), len(yh) - yh.sum()) >= 5:
    ph = {"quality_only": cv_prob(hq, [Q], C=1.0), "gaze_only": cv_prob(hq, GAZE)}
    for name, p in ph.items():
        rows.append({"analysis": "tracking ratio >= median", "model": name, "n": len(yh),
                     "auroc": U.fmt_ci(U.bootstrap_ci(yh, p, n_boot=N_BOOT))})
    tests.append({"model": "gaze_only (high-quality)", "vs": "quality_only (high-quality)",
                  **U.delong_test(yh, ph["gaze_only"], ph["quality_only"])})
res = pd.DataFrame(rows)
print(U.table_md(res))
res.to_csv(OUT / "gaze_quality_auroc.csv", index=False)
tests = pd.DataFrame(tests)
tests["p_holm"] = U.holm(tests.p.values)
print(U.table_md(tests))
tests.to_csv(OUT / "gaze_quality_delong.csv", index=False)


# ## 4. Permutation test (gaze only, and gaze with quality removed)
# 

rng = np.random.default_rng(0)
perm = {}
for name, resid in (("gaze_only", False), ("gaze_quality_residualised", True)):
    obs = U._safe_auc(y, probs[name])
    null = np.array([U._safe_auc(lab, cv_prob(data, GAZE, resid=resid, repeats=1, labels=lab))
                     for lab in (rng.permutation(y) for _ in range(N_PERM))])
    perm[name] = {"auroc": obs, "null_mean": float(null.mean()), "null_95": float(np.quantile(null, 0.95)),
                  "p_perm": float((1 + (null >= obs).sum()) / (1 + N_PERM)), "n_perm": N_PERM}
    print(name, perm[name])
U.save_json(perm, OUT / "gaze_permutation_test.json")


# ## 5. Gaze features vs tracking ratio
# 

cor = pd.DataFrame([{"feature": c, "spearman_with_quality": data[[c, Q]].corr("spearman").iloc[0, 1],
                     "spearman_with_label": data[[c, "label"]].corr("spearman").iloc[0, 1]} for c in GAZE])
cor = cor.reindex(cor.spearman_with_quality.abs().sort_values(ascending=False).index)
print(U.table_md(cor.head(15)))
cor.to_csv(OUT / "gaze_feature_quality_correlation.csv", index=False)
print("label vs tracking ratio (Spearman):", round(data[[Q, "label"]].corr("spearman").iloc[0, 1], 3))


In [ ]:
R.run('g1')

In [ ]:
R.finish()